# Phase 2: Data Understanding & Data Profiling
## E-Commerce Sales & Customer Analytics 

### 1. Struktur Dataset (3 Tabel Utama)
Proyek ini mengintegrasikan 3 dataset relasional e-commerce:
1. **`ecommerce_sales_customer_analytics_150k.csv` (Tabel Transaksi / Fact Table):** Berisi 138.116 baris riwayat transaksi pesanan, detail biaya, pendapatan, diskon, dan status pengiriman.
2. **`customer_master.csv` (Dimensi Pelanggan / Customer Dimension):** Berisi 25.000 data profil unik pelanggan (demografi, lokasi, segmen, dan biaya akuisisi).
3. **`product_catalog.csv` (Dimensi Produk / Product Dimension):** Berisi 1.175 data katalog produk (kategori, subkategori, merek, harga satuan, dan biaya produk).

### 2. Fokus Bisnis Utama (*The Golden Trio*)
Tiga pertanyaan bisnis paling krusial yang dijawab pada tahap ini:
1. **Profitabilitas & Diskon:** Apakah pemberian diskon berhasil meningkatkan penjualan, atau justru menggerus profitabilitas perusahaan?
2. **Anomali Transaksi (*High Revenue, Low Margin*):** Transaksi mana yang omzetnya tinggi tapi marginnya tipis sehingga membebani operasional dan gudang?
3. **Kontribusi Segmen Pelanggan:** Segmen pelanggan mana yang paling menghasilkan profit bersih sehingga layak mendapat alokasi anggaran pemasaran terbesar?

### 3. Mengapa Tahap *Data Profiling* Sangat Penting?
Data profiling adalah fondasi utama sebelum data masuk ke tahap visualisasi (*Dashboard*) atau *Machine Learning*:
* **Mencegah Prinsip *Garbage In, Garbage Out*:** Jika data kotor atau banyak anomali, model prediksi dan dashboard akan menghasilkan kesimpulan yang menyesatkan.
* **Menjamin Validitas Finansial:** Menghindari salah hitung profit, omzet, atau kerugian akibat data duplikat atau rumus yang tidak konsisten.

### Alur Pengerjaan di Notebook Ini:
1. **Setup Lingkungan & Library:** Memuat library analitik dan mengatur format tampilan angka.
2. **Pengecekan Struktur Awal:** Membaca sampel kecil untuk memahami korelasi antar-tabel.
3. **Optimasi Memori (Downcasting & Categorical):** Menghemat penggunaan RAM sekitar 54% saat memuat seluruh data (206,2 MB menjadi 94,0 MB).
4. **Validasi Kunci Utama (Completeness & Uniqueness):** Memastikan ID transaksi dan pelanggan 100% lengkap tanpa duplikat.
5. **Targeted Profiling (*The Golden Trio*):** Menyelidiki diskon, pesanan rugi, anomali margin, dan kontribusi segmen.
6. **Pemeriksaan Kesehatan Data Umum:** Memeriksa missing values kritis dan konsistensi hitungan finansial.
7. **Executive Summary & Rekomendasi:** Merangkum temuan penting dan langkah tindak lanjut bisnis.

## 1. Import Libraries & Setup Lingkungan Analisis

**Tujuan:**
Memuat pustaka (*libraries*) utama Python untuk analisis data numerik dan tabular, serta mengatur format tampilan output di Jupyter Notebook.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Display settings untuk output yang lebih rapi di Jupyter
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

print(" Libraries loaded successfully")
print(f" Pandas version: {pd.__version__}")
print(f" NumPy version: {np.__version__}")

 Libraries loaded successfully
 Pandas version: 3.0.6
 NumPy version: 2.5.3


## 2. Pengecekan Struktur Awal Data 

**Strategi:**
Sebelum memuat seluruh file berukuran besar ke memori, kita membaca **sampel kecil (5 baris pertama)** terlebih dahulu.

**Tujuan Utama:**
1. **Memeriksa Daftar Kolom:** Memastikan nama-nama kolom yang tersedia di masing-masing tabel (`sales`, `customers`, `products`).
2. **Menganalisis Kebutuhan Penggabungan (*Join / Merge*):**
   * Apakah tabel transaksi (`sales`) sudah memiliki informasi produk (kategori, harga modal, merek) secara langsung?
   * Apakah tabel transaksi memiliki kolom `product_id` untuk menghubungkan ke `product_catalog`?
   * Apakah tabel transaksi dapat langsung dianalisis atau memerlukan tabel perantara (*bridge table*)?

In [2]:
print("Initial Structure Check")

sales_sample = pd.read_csv('data/source/ecommerce_sales_customer_analytics_150k.csv', nrows=5)
customers_sample = pd.read_csv('data/source/customer_master.csv', nrows=5)
products_sample = pd.read_csv('data/source/product_catalog.csv', nrows=5)

print("\n--- Sales Table Columns ---")
print(f"Total columns: {len(sales_sample.columns)}")
print(sales_sample.columns.tolist())

print("\n--- Customer Master Columns ---")
print(f"Total columns: {len(customers_sample.columns)}")
print(customers_sample.columns.tolist())

print("\n--- Product Catalog Columns ---")
print(f"Total columns: {len(products_sample.columns)}")
print(products_sample.columns.tolist())

# Check for product details in sales table
print("Merge Requirement Analysis")
has_product_category = 'product_category' in sales_sample.columns
has_product_cost_sales = 'product_cost' in sales_sample.columns
has_brand = 'brand' in sales_sample.columns

print(f"\nSales table has product_category? {has_product_category}")
print(f"Sales table has product_cost? {has_product_cost_sales}")
print(f"Sales table has brand? {has_brand}")

# Check merge keys
has_product_id_sales = 'product_id' in sales_sample.columns
print(f"\nSales table has product_id? {has_product_id_sales}")

if has_product_id_sales:
    print("Sales table can be directly linked to product_catalog")
else:
    print("Sales table does not have product_id - may need order_items bridge table")

Initial Structure Check

--- Sales Table Columns ---
Total columns: 46
['order_id', 'order_date', 'order_time', 'order_status', 'sales_channel', 'customer_id', 'customer_name', 'customer_age', 'gender', 'customer_segment', 'customer_type', 'customer_city', 'customer_state', 'customer_country', 'region', 'customer_postal_code', 'payment_method', 'payment_status', 'currency', 'shipping_method', 'warehouse', 'delivery_days', 'estimated_delivery_days', 'delivery_status', 'return_status', 'return_reason', 'customer_rating', 'review_sentiment', 'customer_review', 'marketing_channel', 'campaign_name', 'coupon_code', 'loyalty_points_earned', 'loyalty_points_redeemed', 'quantity', 'gross_sales', 'discount_amount', 'tax_amount', 'shipping_cost', 'net_sales', 'product_cost', 'profit', 'profit_margin_percentage', 'customer_lifetime_value', 'is_repeat_customer', 'customer_order_count']

--- Customer Master Columns ---
Total columns: 11
['customer_id', 'customer_name', 'customer_age', 'gender', 'cus

## 3. Pemuatan Seluruh Data dengan Optimasi Memori

**Mengapa Optimasi Memori Diperlukan?**
Dataset transaksi memiliki 138.116 baris dan 46 kolom. Jika dimuat dengan tipe data bawaan Pandas, penggunaan RAM terukur **206,2 MB**. Dengan optimasi, penggunaan turun menjadi **94,0 MB (turun 54,4%)**, membuat proses analisis lebih ringan. Angka baseline diukur dengan `pd.read_csv` tanpa penyesuaian tipe dan `memory_usage(deep=True)`.

**Teknik Penentuan Tipe Data (*Data Types*):**
1. **`category`:** Digunakan untuk kolom teks dengan variasi nilai unik terbatas (contoh: `gender`, `order_status`, `payment_method`, `customer_segment`). Teks hanya disimpan satu kali di memori dan dipetakan dengan integer, menghemat 70–90% RAM.
2. **`int8` & `int16`:** Menggantikan `int64` untuk angka bulat kecil.
   * `int8` (rentang -128 s.d. 127) sangat cukup untuk `customer_age`.
   * `int16` (rentang -32.768 s.d. 32.767) cocok untuk `quantity` dan poin loyalitas.
3. **`float64` untuk kolom keuangan:** kolom finansial dan `profit_margin_percentage` dibaca sebagai `float64`. `float32` hemat memori, tetapi jumlah agregatnya meleset beberapa dolar (pada kolom sumber `net_sales`, `float32` memberi 177.134.272,00 vs `float64` 177.134.263,74), sehingga angka yang dilaporkan memakai `float64`. Kolom desimal non-finansial (`customer_rating`, `delivery_days`) tetap `float32`.
4. **`parse_dates=['order_date']`:** Mengonversi kolom tanggal ke format `datetime64` agar langsung siap untuk analisis tren waktu.

In [3]:
print("LOADING FULL DATASETS WITH MEMORY OPTIMIZATION")

# Dtype mapping untuk sales dataset
dtype_sales = {
    'order_id': 'str',
    'customer_id': 'str',
    'customer_name': 'str',
    'gender': 'category',
    'customer_segment': 'category',
    'customer_type': 'category',
    'customer_city': 'str',
    'customer_state': 'str',
    'customer_country': 'category',
    'region': 'category',
    'order_status': 'category',
    'sales_channel': 'category',
    'payment_method': 'category',
    'payment_status': 'category',
    'currency': 'category',
    'shipping_method': 'category',
    'warehouse': 'category',
    'delivery_status': 'category',
    'return_status': 'category',
    'return_reason': 'str',
    'review_sentiment': 'category',
    'marketing_channel': 'category',
    'campaign_name': 'str',
    'coupon_code': 'str',
    'customer_postal_code': 'str',
    'customer_review': 'str',
    'customer_age': 'int8',
    'quantity': 'int16',
    'loyalty_points_earned': 'int16',
    'loyalty_points_redeemed': 'int16',
    'customer_order_count': 'int16',
    'delivery_days': 'float32',
    'estimated_delivery_days': 'float32',
    'customer_rating': 'float32',
    # Kolom keuangan memakai float64: float32 membuat total agregat meleset beberapa dolar.
    'gross_sales': 'float64',
    'discount_amount': 'float64',
    'tax_amount': 'float64',
    'shipping_cost': 'float64',
    'net_sales': 'float64',
    'product_cost': 'float64',
    'profit': 'float64',
    'profit_margin_percentage': 'float64',
    'customer_lifetime_value': 'float32',
    'is_repeat_customer': 'bool'
}

# Dtype mapping untuk customer master
dtype_customers = {
    'customer_id': 'str',
    'customer_name': 'str',
    'gender': 'category',
    'customer_segment': 'category',
    'customer_city': 'str',
    'customer_state': 'str',
    'customer_country': 'category',
    'region': 'category',
    'customer_postal_code': 'str',
    'customer_age': 'int8',
    'customer_acquisition_cost': 'float32'
}

# Dtype mapping untuk product catalog
dtype_products = {
    'product_id': 'str',
    'product_name': 'str',
    'product_category': 'category',
    'product_subcategory': 'category',
    'brand': 'category',
    'supplier': 'category',
    'unit_price': 'float32',
    'product_cost': 'float32',
    'product_rating': 'float32'
}

df_sales = pd.read_csv(
    'data/source/ecommerce_sales_customer_analytics_150k.csv', 
    dtype=dtype_sales,
    parse_dates=['order_date']
)

df_customers = pd.read_csv(
    'data/source/customer_master.csv', 
    dtype=dtype_customers
)

df_products = pd.read_csv(
    'data/source/product_catalog.csv', 
    dtype=dtype_products
)

# Memory usage comparison
mem_sales = df_sales.memory_usage(deep=True).sum() / 1024**2
mem_customers = df_customers.memory_usage(deep=True).sum() / 1024**2
mem_products = df_products.memory_usage(deep=True).sum() / 1024**2

print(f"\n Sales Dataset: {df_sales.shape} - Memory: {mem_sales:.2f} MB")
print(f" Customer Master: {df_customers.shape} - Memory: {mem_customers:.2f} MB")
print(f" Product Catalog: {df_products.shape} - Memory: {mem_products:.2f} MB")
print(f"\nTotal Memory Usage: {mem_sales + mem_customers + mem_products:.2f} MB")

print(f"\nDate Range: {df_sales['order_date'].min().date()} to {df_sales['order_date'].max().date()}")
print(f"Total Transactions: {len(df_sales):,}")
print(f"Unique Customers: {df_sales['customer_id'].nunique():,}")


# --- KPI basis locked (decision D7) -----------------------------------------
# Overwrite the source financial columns with the approved KPI basis so every
# downstream cell uses it. Source values are re-read in cell 20 for the as-stored check.
df_sales['net_sales'] = (df_sales['gross_sales'] - df_sales['discount_amount']).astype('float64')
df_sales['profit'] = (df_sales['net_sales'] - df_sales['product_cost']).astype('float64')
df_sales['profit_margin_percentage'] = (df_sales['profit'] / df_sales['net_sales'] * 100).astype('float64')

print(f"\nKPI basis locked (D7): net_sales = gross_sales - discount_amount; profit = net_sales - product_cost")
print(f"  Net sales (basis C): ${df_sales['net_sales'].sum():,.2f}")
print(f"  Profit (basis C):    ${df_sales['profit'].sum():,.2f}")
print(f"  Aggregate margin:    {df_sales['profit'].sum() / df_sales['net_sales'].sum() * 100:.2f}%")
print(f"  Pass-through tax:    ${df_sales['tax_amount'].sum():,.2f}")
print(f"  Pass-through shipping: ${df_sales['shipping_cost'].sum():,.2f}")


LOADING FULL DATASETS WITH MEMORY OPTIMIZATION

 Sales Dataset: (138116, 46) - Memory: 94.00 MB
 Customer Master: (25000, 11) - Memory: 7.26 MB
 Product Catalog: (1175, 9) - Memory: 0.23 MB

Total Memory Usage: 101.49 MB

Date Range: 2021-01-01 to 2025-12-31
Total Transactions: 138,116
Unique Customers: 24,911

KPI basis locked (D7): net_sales = gross_sales - discount_amount; profit = net_sales - product_cost
  Net sales (basis C): $157,143,214.06
  Profit (basis C):    $59,232,305.35
  Aggregate margin:    37.69%
  Pass-through tax:    $16,914,090.41
  Pass-through shipping: $3,076,959.27


## 4. Validasi Kelengkapan & Keunikan Kunci Utama 

**Alasan Kunci Utama (*Primary Key*) Wajib 100% Lengkap dan Unik**

Kunci identifikasi adalah fondasi kebenaran data bisnis:
1. **`order_id` (ID Transaksi):** Tanpa ini, transaksi tidak dapat dilacak ke sistem keuangan/logistik. Jika duplikat, omzet akan terhitung ganda (*double-counting*).
2. **`customer_id` (ID Pelanggan):** Tanpa ini, transaksi tidak bisa dihubungkan ke profil pembeli sehingga segmentasi dan analisis retensi menjadi bias.
3. **`product_id` (ID Produk):** Tanpa ini, performa produk dan pengelolaan persediaan (*inventory*) tidak dapat diukur.

**Kriteria Keberhasilan (*Acceptance Criteria*):**
* **Kelengkapan (*Completeness*) = 100%:** Tidak boleh ada nilai kosong (*Missing Value / NaN*) pada kolom ID.
* **Keunikan (*Uniqueness*) = 0 Duplikat:** Setiap ID hanya boleh muncul satu kali pada tabel master agar perhitungan metrik valid.

In [4]:
print("Data Completeness: Primary Keys")

# Check completeness of critical IDs
total_records = len(df_sales)

order_id_complete = df_sales['order_id'].notna().sum()
customer_id_complete = df_sales['customer_id'].notna().sum()

order_id_pct = (order_id_complete / total_records) * 100
customer_id_pct = (customer_id_complete / total_records) * 100

print(f"\n[CRITICAL CHECK] Order_ID Completeness:")
print(f"  Complete: {order_id_complete:,} / {total_records:,} ({order_id_pct:.2f}%)")
if order_id_pct == 100:
    print(" PASS - All orders have Order_ID")
else:
    print(f" FAIL - Missing {total_records - order_id_complete:,} Order_IDs")

print(f"\n[CRITICAL CHECK] Customer_ID Completeness:")
print(f"  Complete: {customer_id_complete:,} / {total_records:,} ({customer_id_pct:.2f}%)")
if customer_id_pct == 100:
    print("  PASS - All orders have Customer_ID")
else:
    print(f"  FAIL - Missing {total_records - customer_id_complete:,} Customer_IDs")

# Cek apakah produk_id ada di sales tabel 
if 'product_id' in df_sales.columns:
    product_id_complete = df_sales['product_id'].notna().sum()
    product_id_pct = (product_id_complete / total_records) * 100
    print(f"\n[CRITICAL CHECK] Product_ID Completeness:")
    print(f"  Complete: {product_id_complete:,} / {total_records:,} ({product_id_pct:.2f}%)")
    if product_id_pct == 100:
        print("  PASS - All orders have Product_ID")
    else:
        print(f" FAIL - Missing {total_records - product_id_complete:,} Product_IDs")
else:
    print(f"\n[INFO] Product_ID not in sales table - product details at aggregate level")

# Cek duplikat primary key atau ID pada setiap tabel
print("UNIQUENESS CHECK: Primary Keys")
dup_orders = df_sales.duplicated(subset=['order_id'], keep=False).sum()
dup_customers_master = df_customers.duplicated(subset=['customer_id'], keep=False).sum()
dup_products = df_products.duplicated(subset=['product_id'], keep=False).sum()

print(f"\nDuplicate Order_IDs in Sales: {dup_orders}")
if dup_orders == 0:
    print("  PASS - All Order_IDs are unique")
else:
    print(f"  FAIL - Found {dup_orders} duplicate orders")

print(f"\nDuplicate Customer_IDs in Master: {dup_customers_master}")
if dup_customers_master == 0:
    print("  PASS - All Customer_IDs are unique")
else:
    print(f"  FAIL - Found {dup_customers_master} duplicate customers")

print(f"\nDuplicate Product_IDs in Catalog: {dup_products}")
if dup_products == 0:
    print("  PASS - All Product_IDs are unique")
else:
    print(f"  FAIL - Found {dup_products} duplicate products")

Data Completeness: Primary Keys

[CRITICAL CHECK] Order_ID Completeness:
  Complete: 138,116 / 138,116 (100.00%)
 PASS - All orders have Order_ID

[CRITICAL CHECK] Customer_ID Completeness:
  Complete: 138,116 / 138,116 (100.00%)
  PASS - All orders have Customer_ID

[INFO] Product_ID not in sales table - product details at aggregate level
UNIQUENESS CHECK: Primary Keys

Duplicate Order_IDs in Sales: 0
  PASS - All Order_IDs are unique

Duplicate Customer_IDs in Master: 0
  PASS - All Customer_IDs are unique

Duplicate Product_IDs in Catalog: 0
  PASS - All Product_IDs are unique


## 5. Menganalisis 3 Pilar Utama Bisnis


### 5.1 Profitabilitas & Evaluasi Diskon

**Konteks Bisnis:**
Diskon adalah strategi pemasaran yang ibarat **pedang bermata dua**: efektif menarik pembeli, tetapi berisiko menggerus laba bersih jika tidak dikontrol dengan baik.

**Fokus Pemeriksaan di Sel Ini:**
1. **Anomali Diskon (>100%):** Mendeteksi apakah ada diskon yang melebihi harga kotor (*gross sales*), indikasi kesalahan sistem input.
2. **Distribusi Persentase Diskon:** Melihat rata-rata, nilai tengah, dan seberapa banyak transaksi yang mendapat diskon ekstrem (>50%).
3. **Kesehatan Profitabilitas (*Loss-Making Orders*):** Menghitung berapa banyak pesanan yang menghasilkan **profit negatif (rugi)** dan berapa total kerugian yang ditanggung perusahaan.
4. **Distribusi Margin Keuntungan:** Mengetahui apakah margin profit bisnis secara keseluruhan tergolong sehat (>40%) atau tipis (<20%).

In [5]:
print(" Profitabilitas & Diskon")


# Calculate discount rate (safe division, float64 for accurate bracket edges)
df_sales['discount_rate'] = np.where(
    df_sales['gross_sales'] > 0,
    df_sales['discount_amount'] / df_sales['gross_sales'] * 100,
    0
).astype('float64')

# Check discount anomalies
# To check if there are any discounts exceeding 100%
discount_over_100 = df_sales[df_sales['discount_amount'] > df_sales['gross_sales']]
print(f"\n--- Discount Anomalies --- ")
print(f"Discounts > Gross Sales (>100%): {len(discount_over_100)} orders ({len(discount_over_100)/len(df_sales)*100:.2f}%)")
if len(discount_over_100) > 0:
    print("  WARNING - Impossible discounts detected! Data quality issue.")
else:
    print(" No impossible discounts")

# Discount distribution
# To check mean, median, quantile, and max distribution discount
print(f"\n--- Discount Rate Distribution ---")
print(f"Mean: {df_sales['discount_rate'].mean():.2f}%")
print(f"Median: {df_sales['discount_rate'].median():.2f}%")
print(f"75th Percentile: {df_sales['discount_rate'].quantile(0.75):.2f}%")
print(f"Max: {df_sales['discount_rate'].max():.2f}%")

heavy_discount = df_sales[df_sales['discount_rate'] > 50]
print(f"\nOrders with >50% discount: {len(heavy_discount)} ({len(heavy_discount)/len(df_sales)*100:.2f}%)")
print(f"Revenue from heavy discounts: ${heavy_discount['net_sales'].sum():,.2f}")

# Negative profit analysis
negative_profit = df_sales[df_sales['profit'] < 0]
print(f"\n--- Profit Health Check ---")
print(f"Negative Profit Orders: {len(negative_profit)} ({len(negative_profit)/len(df_sales)*100:.2f}%)")
print(f"Total Loss from Negative Profit: ${negative_profit['profit'].sum():,.2f}")
if len(negative_profit) > 0:
    print(f"Average Loss per Order: ${negative_profit['profit'].mean():,.2f}")

# Overall profitability
print(f"\n--- Overall Profitability Metrics ---")
print(f"Total Revenue: ${df_sales['net_sales'].sum():,.2f}")
print(f"Total Profit: ${df_sales['profit'].sum():,.2f}")
print(f"Overall Profit Margin (aggregate): {(df_sales['profit'].sum() / df_sales['net_sales'].sum() * 100):.2f}%")
print(f"Average Transaction Margin: {df_sales['profit_margin_percentage'].mean():.2f}%")
print(f"\nProfit Margin Distribution:")
print(f"  Mean: {df_sales['profit_margin_percentage'].mean():.2f}%")
print(f"  Median: {df_sales['profit_margin_percentage'].median():.2f}%")
print(f"  25th Percentile: {df_sales['profit_margin_percentage'].quantile(0.25):.2f}%")
print(f"  75th Percentile: {df_sales['profit_margin_percentage'].quantile(0.75):.2f}%")

 Profitabilitas & Diskon

--- Discount Anomalies --- 
Discounts > Gross Sales (>100%): 0 orders (0.00%)
 No impossible discounts

--- Discount Rate Distribution ---
Mean: 15.12%
Median: 12.75%
75th Percentile: 20.60%
Max: 60.02%

Orders with >50% discount: 1729 (1.25%)
Revenue from heavy discounts: $1,449,419.53

--- Profit Health Check ---
Negative Profit Orders: 2530 (1.83%)
Total Loss from Negative Profit: $-426,306.80
Average Loss per Order: $-168.50

--- Overall Profitability Metrics ---
Total Revenue: $157,143,214.06
Total Profit: $59,232,305.35
Overall Profit Margin (aggregate): 37.69%
Average Transaction Margin: 41.16%

Profit Margin Distribution:
  Mean: 41.16%
  Median: 43.30%
  25th Percentile: 34.12%
  75th Percentile: 51.04%


### 5.2 Anomali Transaksi (High Ticket-Size, Low Margin)

Transaksi dengan nilai belanja tinggi (High Ticket-Size) sering terlihat sangat bagus. Namun, jika transaksi tersebut memiliki margin keuntungan yang sangat tipis (Low Margin), transaksi itu sebenarnya menguras sumber daya perusahaan

**Contoh:**
* **Produk A:** Omzet $100.000 dengan Margin 50% ➔ Menghasilkan Profit **$50.000** (Sangat Sehat).
* **Produk B:** Omzet $100.000 dengan Margin 5% ➔ Hanya menghasilkan Profit **$5.000** (Beban Operasional Tinggi).

**Kriteria Anomali yang Digunakan:**
* **High Revenue:** Transaksi pada atau di atas kuartil ke-3 (`net_sales >= persentil ke-75`).
* **Low Margin:** Transaksi pada atau di bawah kuartil ke-1 (`profit_margin_percentage <= persentil ke-25`).
* **Operator:** inklusif (`>=` dan `<=`) sebagai default MVP. Varian ketat (`>` dan `<`) menghasilkan 12.855 transaksi juga; tidak ada transaksi yang tepat berada di ambang di antara kedua aturan.
* **Simulasi kontrafaktual:** selisih ke margin rata-rata dilaporkan sebagai simulasi ilustratif dengan asumsi eksplisit, bukan prakiraan.

In [6]:
print("Anomali Transaksi: High revenue, low margin")

# Untuk mengecek harga produk yang negatif (data quality)
negative_cost = df_sales[df_sales['product_cost'] < 0]
print(f"\n--- Data Quality Check ---")
print(f"Negative Product Costs: {len(negative_cost)} records")
if len(negative_cost) > 0:
    print(f"  Warning - Negative costs impossible! Data issue.")
else:
    print("  No negative costs")

# High revenue, low margin (MVP default: inclusive operators)
revenue_75th = df_sales['net_sales'].quantile(0.75)
margin_25th = df_sales['profit_margin_percentage'].quantile(0.25)

high_rev_low_margin = df_sales[
    (df_sales['net_sales'] >= revenue_75th) &
    (df_sales['profit_margin_percentage'] <= margin_25th)
]
strict_count = int(((df_sales['net_sales'] > revenue_75th) &
                    (df_sales['profit_margin_percentage'] < margin_25th)).sum())

print(f"\n--- Anomaly Definition (inclusive, MVP default) ---")
print(f"High Revenue Threshold (75th percentile): ${revenue_75th:,.2f}")
print(f"Low Margin Threshold (25th percentile): {margin_25th:.2f}%")
print(f"Strict variant gives {strict_count:,} orders; tie-breaking difference = {len(high_rev_low_margin) - strict_count:,} order(s).")

print(f"\n--- Anomaly Orders: High Revenue but Low Margin ---")
print(f"Count: {len(high_rev_low_margin)} orders ({len(high_rev_low_margin)/len(df_sales)*100:.2f}% of total)")
print(f"Total Revenue: ${high_rev_low_margin['net_sales'].sum():,.2f}")
print(f"Total Profit: ${high_rev_low_margin['profit'].sum():,.2f}")
print(f"Average Margin (transaction): {high_rev_low_margin['profit_margin_percentage'].mean():.2f}%")
print(f"Aggregate Margin: {(high_rev_low_margin['profit'].sum() / high_rev_low_margin['net_sales'].sum() * 100):.2f}%")

# Counterfactual simulation (illustrative only, explicit assumptions)
avg_margin = df_sales['profit_margin_percentage'].mean()
agg_margin = df_sales['profit'].sum() / df_sales['net_sales'].sum() * 100
potential_profit = high_rev_low_margin['net_sales'].sum() * (avg_margin / 100)
actual_profit = high_rev_low_margin['profit'].sum()
missed_profit = potential_profit - actual_profit
missed_profit_agg = high_rev_low_margin['net_sales'].sum() * (agg_margin / 100) - actual_profit

print(f"\n--- Counterfactual Simulation (illustrative, NOT a forecast) ---")
print("Assumption: anomaly orders reach the reference margin without changing revenue.")
print("Ignored: demand response, fixed costs, cannibalisation, operational feasibility.")
print(f"Reference average transaction margin: {avg_margin:.2f}%")
print(f"  Potential Profit: ${potential_profit:,.2f}")
print(f"  Actual Profit: ${actual_profit:,.2f}")
print(f"  Gap (average transaction margin basis): ${missed_profit:,.2f}")
print(f"Reference aggregate margin: {agg_margin:.2f}%")
print(f"  Gap (aggregate margin basis): ${missed_profit_agg:,.2f}")

# Customer segment mana yang menjadi penyebabnya
print(f"\n--- Which Segments Have This Problem? ---")
anomaly_by_segment = high_rev_low_margin.groupby('customer_segment').agg({
    'order_id': 'count',
    'net_sales': 'sum',
    'profit': 'sum',
    'profit_margin_percentage': 'mean'
})
anomaly_by_segment.columns = ['Order_Count', 'Revenue', 'Profit', 'Avg_Margin']
print(anomaly_by_segment.sort_values('Revenue', ascending=False))


Anomali Transaksi: High revenue, low margin

--- Data Quality Check ---
Negative Product Costs: 0 records
  No negative costs

--- Anomaly Definition (inclusive, MVP default) ---
High Revenue Threshold (75th percentile): $1,552.32
Low Margin Threshold (25th percentile): 34.12%
Strict variant gives 12,855 orders; tie-breaking difference = 0 order(s).

--- Anomaly Orders: High Revenue but Low Margin ---
Count: 12855 orders (9.31% of total)
Total Revenue: $34,665,272.99
Total Profit: $7,528,760.49
Average Margin (transaction): 21.47%
Aggregate Margin: 21.72%

--- Counterfactual Simulation (illustrative, NOT a forecast) ---
Assumption: anomaly orders reach the reference margin without changing revenue.
Ignored: demand response, fixed costs, cannibalisation, operational feasibility.
Reference average transaction margin: 41.16%
  Potential Profit: $14,266,936.03
  Actual Profit: $7,528,760.49
  Gap (average transaction margin basis): $6,738,175.54
Reference aggregate margin: 37.69%
  Gap (ag

### 5.3 Analisis Segmen Pelanggan & Kontribusi Keuntungan

**Prinsip Utama: Tidak Semua Pelanggan Menghasilkan Dampak yang Sama (*Prinsip Pareto 80/20*)**
Dalam bisnis, sebagian kecil segmen pelanggan sering kali menyumbang porsi profit terbesar. Mengetahui karakteristik tiap segmen sangat penting untuk menentukan alokasi anggaran pemasaran secara tepat sasaran.

**Fokus Evaluasi di Sel Ini:**
1. **Kontribusi Omzet vs Profit:** Membandingkan kontribusi pendapatan kotor/bersih dengan kontribusi laba riil tiap segmen (misal: *Consumer*, *Business*, *Premium*, *VIP*).
2. **Strategi Diskon per Segmen:** Mengecek apakah segmen tertentu (seperti VIP) menerima diskon terlalu besar yang tidak sebanding dengan kontribusi keuntungannya.
3. **Efisiensi Segmen (*Profit per Customer* & *Average Order Value*):** Mengukur segmen mana yang paling menguntungkan per kepala pelanggan untuk optimalisasi program loyalitas.

In [7]:
print(" Segmen Pelanggan & Kontribusi Profit ")

# Untuk mengecek kategori segment & Penyebarannya 
print(f"\n--- Segment Overview ---")
print(f"Unique Segments: {df_sales['customer_segment'].nunique()}")
print(f"Segment Values: {sorted(df_sales['customer_segment'].unique().tolist())}")
print(f"\nSegment Distribution:")
print(df_sales['customer_segment'].value_counts())

# Analisis keuntungan per segmen pelanggan
segment_analysis = df_sales.groupby('customer_segment').agg({
    'order_id': 'count',
    'customer_id': 'nunique',
    'net_sales': 'sum',
    'gross_sales': 'sum',
    'discount_amount': 'sum',
    'product_cost': 'sum',
    'profit': 'sum',
    'profit_margin_percentage': 'mean',
    'customer_lifetime_value': 'mean'
}).round(2)

segment_analysis.columns = ['Order_Count', 'Unique_Customers', 'Net_Revenue', 
                            'Gross_Revenue', 'Total_Discounts', 'Total_Cost',
                            'Total_Profit', 'Avg_Margin_%', 'Avg_CLV']

# Hitung persentase kontribusi revenue dan profit
segment_analysis['Revenue_Contribution_%'] = (
    segment_analysis['Net_Revenue'] / segment_analysis['Net_Revenue'].sum() * 100
).round(2)

segment_analysis['Profit_Contribution_%'] = (
    segment_analysis['Total_Profit'] / segment_analysis['Total_Profit'].sum() * 100
).round(2)

# Hitung rata-rata nilai per pesanan
segment_analysis['Avg_Order_Value'] = (
    segment_analysis['Net_Revenue'] / segment_analysis['Order_Count']
).round(2)

# Urutkan dari tertinggi ke yang terendah
segment_analysis = segment_analysis.sort_values('Total_Profit', ascending=False)

print(f"\n--- Segment Profitability Analysis ---")
print(segment_analysis[['Order_Count', 'Unique_Customers', 'Net_Revenue', 'Total_Profit', 
                        'Avg_Margin_%', 'Revenue_Contribution_%', 'Profit_Contribution_%', 
                        'Avg_Order_Value']])

# Analisis strategi diskon per segmen 
print(f"\n--- Discount Strategy by Segment ---")
segment_discount = df_sales.groupby('customer_segment').agg({
    'discount_rate': 'mean',
    'discount_amount': 'mean'
}).round(2)
segment_discount.columns = ['Avg_Discount_Rate_%', 'Avg_Discount_Amount_$']
print(segment_discount.sort_values('Avg_Discount_Rate_%', ascending=False))

# Hitung rata-rata profit per 1 pelanggan
print(f"\n--- Segment Efficiency: Profit per Customer ---")
segment_analysis['Profit_per_Customer'] = (
    segment_analysis['Total_Profit'] / segment_analysis['Unique_Customers']
).round(2)
print(segment_analysis[['Unique_Customers', 'Total_Profit', 'Profit_per_Customer']].sort_values(
    'Profit_per_Customer', ascending=False
))

 Segmen Pelanggan & Kontribusi Profit 

--- Segment Overview ---
Unique Segments: 4
Segment Values: ['Business', 'Consumer', 'Premium', 'VIP']

Segment Distribution:
customer_segment
Consumer    75636
Premium     34746
VIP         13957
Business    13777
Name: count, dtype: int64

--- Segment Profitability Analysis ---
                  Order_Count  Unique_Customers   Net_Revenue  Total_Profit  \
customer_segment                                                              
Consumer                75636             13585 86,043,495.08 33,556,898.53   
Premium                 34746              6275 39,446,064.06 13,951,730.99   
Business                13777              2515 15,722,434.82  6,088,716.77   
VIP                     13957              2536 15,931,220.10  5,634,959.06   

                  Avg_Margin_%  Revenue_Contribution_%  Profit_Contribution_%  \
customer_segment                                                                
Consumer                 42.56            

## 6. Pemeriksaan Kesehatan Data Menyeluruh
Kita perlu memastikan kualitas data pada seluruh kolom penting lainnya untuk mencegah bias saat pembuatan model prediksi atau visualisasi dashboard.

### 6.1 Analisis Nilai Hilang (Missing Values Analysis)

**Konteks Penting: Tidak Semua Nilai Kosong Merupakan Masalah**
* **Wajar (*By Design*):** 
  * Kolom `return_reason` kosong jika pembeli memang tidak meretur barang.
  * Kolom `coupon_code` kosong jika transaksi tidak menggunakan kode promo.
* **Kritis (*Data Issue*):**
  * Kolom keuangan seperti `profit`, `net_sales`, `gross_sales`, atau status pesanan bernilai kosong.

**Pedoman Batas Toleransi Missing Values:**
* **< 1% Missing:** Wajar dan aman ditangani dengan imputasi (mengisi rata-rata, median, atau modus).
* **1% – 5% Missing:** Perlu diinvestigasi polanya untuk memastikan tidak ada bias sistemik.
* **> 5% Missing pada Kolom Kunci:** Masalah kritis yang harus ditangani sebelum data digunakan untuk pemodelan.

In [8]:
# Cek missing values
print("General Health: Missing Values")

# Kolom Key Bisnis (Yang harus dianalisis)
key_business_cols = [
    'order_id', 'customer_id', 'order_date', 'order_status',
    'customer_segment', 'customer_country', 'region',
    'gross_sales', 'discount_amount', 'net_sales', 
    'product_cost', 'profit', 'profit_margin_percentage',
    'payment_method', 'payment_status'
]

missing_counts = df_sales[key_business_cols].isnull().sum()
missing_pct = (missing_counts / len(df_sales) * 100).round(2)

missing_df = pd.DataFrame({
    'Column': missing_counts.index,
    'Missing_Count': missing_counts.values,
    'Missing_%': missing_pct.values
})

# Hanya menunjukkan kolom yang nilainya kosong
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values(
    'Missing_%', ascending=False
)

print(f"\n--- Key Business Columns Missing Values ---")
if len(missing_df) > 0:
    print(missing_df.to_string(index=False))
    print(f"\n Found missing values in {len(missing_df)} key columns")
else:
    print(" No missing values in key business columns")

# Cek kelengkapan value pada data customer
print(f"\n--- Customer Master Completeness ---")
customer_missing = df_customers[['customer_id', 'customer_segment', 'customer_country', 
                                  'region', 'customer_acquisition_cost']].isnull().sum()
if customer_missing.sum() > 0:
    print(customer_missing[customer_missing > 0])
else:
    print(" No missing values in customer master key columns")

# Cek kelengkapan value pada data produk
print(f"\n--- Product Catalog Completeness ---")
product_missing = df_products[['product_id', 'product_category', 'brand', 
                                'unit_price', 'product_cost']].isnull().sum()
if product_missing.sum() > 0:
    print(product_missing[product_missing > 0])
else:
    print(" No missing values in product catalog key columns")

General Health: Missing Values

--- Key Business Columns Missing Values ---
 No missing values in key business columns

--- Customer Master Completeness ---
 No missing values in customer master key columns

--- Product Catalog Completeness ---
 No missing values in product catalog key columns


### 6.2 Uji Konsistensi Logika Data

**Mengapa Konsistensi Hitungan Sangat Penting?**
Data transaksi harus mematuhi aturan matematika keuangan yang logis. Jika angka pada database inkonsisten, laporan keuangan yang disajikan ke manajemen akan keliru.

**Aturan Logika yang Diuji:**
1. **Konsistensi Perhitungan Margin:** Memvalidasi apakah nilai `profit_margin_percentage` yang tercatat di dataset benar-benar sesuai dengan rumus matematika:
   $$\text{Profit Margin (\%)} = \left(\frac{\text{Profit}}{\text{Net Sales}}\right) \times 100$$
2. **Ketidakmungkinan Logis (*Logical Impossibilities*):**
   * Mendeteksi jika ada transaksi di mana nilai $\text{Profit} > \text{Net Sales}$ (secara bisnis tidak mungkin laba melebihi omzet bersih).
   * Mendeteksi jika biaya modal produk mencurigakan ($\text{Product Cost} > 2 \times \text{Gross Sales}$).
3. **Pengecekan Rentang Nilai Negatif:** Memastikan tidak ada nilai negatif yang tidak logis pada omzet kotor, omzet bersih, atau harga modal.

In [9]:
print("Data Consistency Checks")

# df_sales carries the KPI basis (D7); re-read the source columns for the as-stored check.
src_sales = pd.read_csv(
    'data/source/ecommerce_sales_customer_analytics_150k.csv',
    usecols=['gross_sales', 'discount_amount', 'tax_amount', 'shipping_cost',
             'net_sales', 'product_cost', 'profit', 'profit_margin_percentage'],
    dtype='float64'
)

# KPI margin consistency
df_sales['calculated_margin'] = (df_sales['profit'] / df_sales['net_sales'] * 100).fillna(0)
margin_diff = (df_sales['calculated_margin'] - df_sales['profit_margin_percentage']).abs()
inconsistent_margin = margin_diff > 0.1

print(f"\n--- Profit Margin Consistency (KPI basis D7) ---")
print(f"Inconsistent calculations: {inconsistent_margin.sum()} orders ({inconsistent_margin.sum()/len(df_sales)*100:.2f}%)")
print(f"  Max difference: {margin_diff.max():.4f}%")
if inconsistent_margin.sum() > 0:
    print("  WARNING - Profit margin calculations inconsistent")
else:
    print("  PASS - Profit margin calculations consistent")

print(f"\n--- Source Financial Formula Checks (as stored) ---")
print("  net_sales = gross_sales - discount_amount + tax_amount + shipping_cost")
print("  profit    = net_sales - product_cost - shipping_cost")

src_net_calc = (src_sales['gross_sales'] - src_sales['discount_amount']
                + src_sales['tax_amount'] + src_sales['shipping_cost'])
print(f"  Net Sales formula: max deviation = ${(src_sales['net_sales'] - src_net_calc).abs().max():.2f}")

src_profit_calc = src_sales['net_sales'] - src_sales['product_cost'] - src_sales['shipping_cost']
print(f"  Profit formula: max deviation = ${(src_sales['profit'] - src_profit_calc).abs().max():.2f}")

print("  [FINDING] Substitution gives profit = gross - discount + tax - product_cost,")
print("            so the collected tax is counted as profit in the source columns.")
print(f"            Tax collected (pass-through) = ${src_sales['tax_amount'].sum():,.2f}")
textbook_gap = (src_sales['net_sales'] - (src_sales['gross_sales'] - src_sales['discount_amount'])).abs()
print(f"  [INFO] Source net_sales vs 'gross - discount': max deviation = ${textbook_gap.max():,.2f}")
print("  [INFO] Textbook Net Sales = Gross - (Discounts + Returns + Allowances) cannot be")
print("         computed: no monetary return/allowance column exists.")

print(f"\n--- Logical Impossibilities (KPI basis) ---")
impossible_profit = df_sales[df_sales['profit'] > df_sales['net_sales']]
print(f"Profit > Net Sales (impossible): {len(impossible_profit)} orders")

impossible_cost = df_sales[df_sales['product_cost'] > df_sales['gross_sales'] * 2]
print(f"Product Cost > 2x Gross Sales (suspicious): {len(impossible_cost)} orders")

Data Consistency Checks



--- Profit Margin Consistency (KPI basis D7) ---
Inconsistent calculations: 0 orders (0.00%)
  Max difference: 0.0000%
  PASS - Profit margin calculations consistent

--- Source Financial Formula Checks (as stored) ---
  net_sales = gross_sales - discount_amount + tax_amount + shipping_cost
  profit    = net_sales - product_cost - shipping_cost
  Net Sales formula: max deviation = $0.00
  Profit formula: max deviation = $0.00
  [FINDING] Substitution gives profit = gross - discount + tax - product_cost,
            so the collected tax is counted as profit in the source columns.
            Tax collected (pass-through) = $16,914,090.41
  [INFO] Source net_sales vs 'gross - discount': max deviation = $2,135.92
  [INFO] Textbook Net Sales = Gross - (Discounts + Returns + Allowances) cannot be
         computed: no monetary return/allowance column exists.

--- Logical Impossibilities (KPI basis) ---
Profit > Net Sales (impossible): 0 orders
Product Cost > 2x Gross Sales (suspicious): 0 o

---
## 7. Ringkasan Eksekutif (Executive Summary: Key Insights)

**Tujuan:**
Mengumpulkan seluruh metrik kunci dari hasil pemeriksaan di atas ke dalam satu ringkasan komprehensif yang siap dipresentasikan kepada manajemen (*stakeholders*).

Ringkasan ini mencakup status kualitas data, ringkasan performa finansial, segmen pelanggan paling potensial, serta identifikasi masalah kebocoran profit (*leakage*).

In [10]:
# Calculate summary metrics
total_revenue = df_sales['net_sales'].sum()
total_profit = df_sales['profit'].sum()
overall_margin = (total_profit / total_revenue * 100)
avg_discount_rate = df_sales['discount_rate'].mean()

top_segment = segment_analysis.index[0]
top_segment_profit_contrib = segment_analysis.iloc[0]['Profit_Contribution_%']
top_segment_margin = segment_analysis.iloc[0]['Avg_Margin_%']

print("Executive Summary: Key Insight")


print(f"""
 DATA QUALITY ASSESSMENT:
     Primary Keys (Order_ID, Customer_ID): {order_id_pct:.1f}% and {customer_id_pct:.1f}% complete
     Duplicate Orders: {dup_orders}
   • Discount Anomalies (>100%): {len(discount_over_100)} orders
   • Negative Profit Orders: {len(negative_profit)} ({len(negative_profit)/len(df_sales)*100:.2f}%)
   • High Revenue, Low Margin: {len(high_rev_low_margin)} orders ({len(high_rev_low_margin)/len(df_sales)*100:.2f}%)
   • Data Consistency Issues: {inconsistent_margin.sum()} margin calculation discrepancies
   • Financial formula checks: source-column formulas verified as stored; KPI uses basis D7

PROFITABILITY OVERVIEW:
   • Total Revenue: ${total_revenue:,.2f}
   • Total Profit: ${total_profit:,.2f}
   • Overall Margin (aggregate): {overall_margin:.2f}%
   • Average Transaction Margin: {df_sales['profit_margin_percentage'].mean():.2f}%
   • Average Discount Rate: {avg_discount_rate:.2f}%
   • Loss from Negative Profit Orders: ${negative_profit['profit'].sum():,.2f}

TOP PERFORMING SEGMENT (by total profit): {top_segment}
   • Profit Contribution: {top_segment_profit_contrib:.2f}%
   • Average Transaction Margin: {top_segment_margin:.2f}%
   • Orders: {segment_analysis.iloc[0]['Order_Count']:,.0f}
   • Unique Customers: {segment_analysis.iloc[0]['Unique_Customers']:,.0f}

HIGH-REVENUE LOW-MARGIN PROBLEM (inclusive rule):
   • Affected Orders: {len(high_rev_low_margin):,} ({len(high_rev_low_margin)/len(df_sales)*100:.2f}%)
   • Revenue Generated: ${high_rev_low_margin['net_sales'].sum():,.2f}
   • Actual Profit: ${actual_profit:,.2f}
   • Average Transaction Margin: {high_rev_low_margin['profit_margin_percentage'].mean():.2f}%
   • Counterfactual Gap (illustrative, NOT a forecast): ${missed_profit:,.2f} at avg transaction margin
   • Counterfactual Gap (aggregate margin basis): ${missed_profit_agg:,.2f}

DATASET OVERVIEW:
   • Total Transactions: {len(df_sales):,}
   • Unique Customers: {df_sales['customer_id'].nunique():,}
   • Date Range: {df_sales['order_date'].min().date()} to {df_sales['order_date'].max().date()}
   • Customer Segments: {df_sales['customer_segment'].nunique()}
""")


Executive Summary: Key Insight

 DATA QUALITY ASSESSMENT:
     Primary Keys (Order_ID, Customer_ID): 100.0% and 100.0% complete
     Duplicate Orders: 0
   • Discount Anomalies (>100%): 0 orders
   • Negative Profit Orders: 2530 (1.83%)
   • High Revenue, Low Margin: 12855 orders (9.31%)
   • Data Consistency Issues: 0 margin calculation discrepancies
   • Financial formula checks: source-column formulas verified as stored; KPI uses basis D7

PROFITABILITY OVERVIEW:
   • Total Revenue: $157,143,214.06
   • Total Profit: $59,232,305.35
   • Overall Margin (aggregate): 37.69%
   • Average Transaction Margin: 41.16%
   • Average Discount Rate: 15.12%
   • Loss from Negative Profit Orders: $-426,306.80

TOP PERFORMING SEGMENT (by total profit): Consumer
   • Profit Contribution: 56.65%
   • Average Transaction Margin: 42.56%
   • Orders: 75,636
   • Unique Customers: 13,585

HIGH-REVENUE LOW-MARGIN PROBLEM (inclusive rule):
   • Affected Orders: 12,855 (9.31%)
   • Revenue Generated: $34,6

## 8. Rekomendasi Bisnis & Rencana Aksi

Berdasarkan hasil data profiling, berikut adalah rekomendasi strategis yang dibagi berdasarkan tingkat urgensinya:

---

### 1. Tindakan Kritis (Perbaiki Segera)
1. **Investigasi Pesanan Rugi (*Negative Profit* - 2.530 Pesanan):**
   - Total kerugian mencapai **$426.306,80**.
   - Identifikasi akar masalah: Apakah akibat diskon yang berlebihan, biaya modal supplier yang terlalu tinggi, atau subsidi ongkos kirim yang tidak tertutupi?
   - *Aksi:* Batasi diskon maksimal atau terapkan harga batas bawah (*minimum floor price*) di sistem untuk mencegah transaksi minus.

---

### 2. Tindakan Penting (Target Kuartal Ini)
2. **Optimalisasi Transaksi *High Revenue, Low Margin*:**
   - Ditemukan **12.855 transaksi (9,31%)** menghasilkan omzet $34,67 Juta tetapi laba bersihnya hanya $7,53 Juta (rata-rata margin transaksi 21,47%).
   - *Dampak (simulasi kontrafaktual, bukan prakiraan):* jika transaksi ini berjalan pada rata-rata margin transaksi perusahaan (41,16%), selisihnya **$6,74 Juta**; pada basis margin agregat (37,69%), selisihnya **$5,54 Juta**. Asumsi: revenue tidak berubah; respons permintaan, biaya tetap, dan kanibalisasi diabaikan.
   - *Aksi:*
     - Negosiasikan kembali harga modal dengan pemasok (*supplier*) untuk kategori produk yang sering masuk anomali ini.
     - Lakukan *bundling* produk bermargin rendah dengan produk aksesoris bermargin tinggi.
3. **Evaluasi Ulang Strategi Diskon Segmen VIP / Premium:**
   - Segmen VIP menerima rata-rata diskon per transaksi 17,8% (lebih tinggi dibanding Consumer 13,7%), namun profit per pelanggannya lebih rendah ($2.222) dibandingkan Consumer ($2.470).
   - *Aksi:* Hitung ulang *Return on Investment* (ROI) program VIP agar diskon yang diberikan sebanding dengan nilai belanja pelanggan.

---

### 3. Tindakan Strategis (Rencana Jangka Panjang)
4. **Perkuat Fokus pada Segmen Pelanggan Utama (*Consumer*):**
   - Segmen *Consumer* menyumbang **56,65% total laba** perusahaan dengan margin agregat 39,00% (rata-rata margin transaksi 42,56%).
   - *Aksi:* Alokasikan porsi anggaran promosi terbesar untuk mempertahankan dan memperluas basis pelanggan segmen ini.
5. **Penerapan *Dynamic Pricing* & Personalisasi Diskon:**
   - Ganti sistem diskon pukul rata dengan diskon dinamis berbasis nilai pelanggan (*Customer Lifetime Value - CLV*) dan riwayat repeat order.

---

### Rencana Tahap Selanjutnya:
- **Phase 3: Exploratory Data Analysis (EDA Mendalam)**
  - Analisis tren waktu & musiman (*seasonality*).
  - Analisis performa wilayah geografis (*regional performance*).
  - Analisis mendalam kategori produk dan perilaku keranjang belanja (*market basket*).

##  Kesimpulan Fase 2: Data Understanding & Profiling

Tahap profiling telah berhasil memvalidasi kesiapan dataset dan mengungkap wawasan bisnis awal yang kritis sebelum melangkah ke analisis eksploratif (EDA). Berikut adalah ringkasan eksekutif dari fase ini:

### 1. Integritas & Kualitas Data
Dataset telah terbukti **bersih, robust, dan aman** untuk digunakan dalam pemodelan maupun visualisasi dashboard:
*   **Kelengkapan & Keunikan:** Kunci utama (`order_id`, `customer_id`) memiliki kelengkapan 100% tanpa ada duplikasi, menghilangkan risiko *double-counting* pada metrik keuangan.
*   **Konsistensi Logika:** Rumus kolom sumber `net_sales = gross_sales - discount_amount + tax_amount + shipping_cost` dan `profit = net_sales - product_cost - shipping_cost` terverifikasi pada semua baris sebagai pembanding "as stored". Basis KPI yang dipakai (keputusan D7) adalah `net_sales = gross_sales - discount_amount` dan `profit = net_sales - product_cost`, dengan pajak dan ongkos kirim ditampilkan terpisah sebagai pass-through. `profit_margin_percentage` konsisten dengan `profit / net_sales` pada basis KPI (selisih maksimum 0,0000%). Tidak ditemukan Profit > Net Sales, biaya modal negatif, atau nilai negatif pada omzet.
*   **Efisiensi Komputasi:** Penerapan *downcasting* integer dan tipe data `category` memangkas penggunaan memori (tabel transaksi ~94 MB), sementara kolom keuangan tetap `float64` agar totalnya presisi.
*   **Status:** Data bebas dari *missing values* kritis dan siap diproses ke Fase 3 (EDA).

### 2. Temuan Strategis Awal (Business Insights)
Profil data berhasil memetakan "The Golden Trio" dan mengidentifikasi area kebocoran profit (*profit leakage*):

*   **Kesehatan Finansial & Diskon:**
    Secara keseluruhan, bisnis memiliki margin agregat (**37,69%**, basis KPI D7). Namun, terdapat kebocoran dari **2.530 pesanan yang merugi** (total kerugian $426.306,80) serta pemberian diskon ekstrem (>50%) pada 1,25% transaksi yang perlu diawasi.
*   **Anomali Transaksi (High Revenue, Low Margin):**
    Ditemukan **12.855 transaksi (9,31%)** yang menciptakan "ilusi kesuksesan". Transaksi ini menghasilkan omzet $34,67 Juta namun hanya menyisakan profit $7,53 Juta (rata-rata margin transaksi 21,47%; margin agregat 21,72%). **Simulasi kontrafaktual (bukan prakiraan):** jika transaksi ini berjalan pada rata-rata margin transaksi perusahaan (41,16%), selisihnya $6,74 Juta; pada basis margin agregat (37,69%), selisihnya $5,54 Juta.
*   **Plot Twist Segmen Pelanggan:**
    Segmen **Consumer** adalah tulang punggung profitabilitas perusahaan (menyumbang **56,65% total profit**, margin agregat 39,00%, rata-rata margin transaksi 42,56%, profit per pelanggan $2.470). Segmen **VIP dan Premium** memberikan profit per pelanggan lebih rendah ($2.222 dan $2.223) dengan rata-rata diskon per transaksi lebih tinggi (17,8% dan 17,7%). Perbedaan margin agregat antar segmen kecil (35,4%-39,0%).
*   **Keterbatasan:** `is_repeat_customer` (True untuk 99,76% transaksi) dan `customer_order_count` belum dapat dipakai untuk analisis retensi; CLV belum tervalidasi; metrik finansial level produk ditunda sampai `order_items.csv` direkonsiliasi. Rincian di `docs/EDA_VALIDATION.md`.
